In [1]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5), # Mean for RGB
        (0.5, 0.5, 0.5), # Std for RGB
    )
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

classes = ['plane','car','bird','cat','deer', 'dog','frog','horse','ship','truck']

In [3]:
print(f"train len: {len(train_dataset)}")
print(f"test len: {len(test_dataset)}")

train len: 50000
test len: 10000


In [4]:
# a = h + 2p - k
# b = a/s
# h_out = b + 1
((32+2-3)/1) + 1

32.0

In [5]:
(32+2-2)/2

16.0

In [6]:
class CIFAR10Net(nn.Module):


    def __init__(self):

        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),            
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 10)
        )


    def forward(self, X):

        features = self.features(X)
        return self.classifier(features)

In [7]:
model = CIFAR10Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [8]:
start = time.time()
epochs = 10
for epoch in range(1, epochs+1):
    for images, labels in train_loader:
    
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    if epoch % 5 == 0:
        print(loss.item())
print(f"total time: {time.time()-start}")

0.6398941278457642
0.16798026859760284
total time: 1920.6774275302887


In [9]:
model.eval()
class_corrected = [0] * 10
class_total = [0] * 100

In [10]:
1900/60

31.666666666666668

In [11]:
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        for i in range(len(labels)):
            label = labels[i].item()
            class_corrected[label] += (preds[i] == labels[i]).item()
            class_total[label] += 1

In [12]:
print("\nPer-class accuracy:")
for i, cls in enumerate(classes):
    acc = 100 * class_corrected[i] / class_total[i]
    print(f"  {cls:8s}: {acc:.1f}%")


Per-class accuracy:
  plane   : 78.3%
  car     : 83.3%
  bird    : 74.1%
  cat     : 65.8%
  deer    : 72.3%
  dog     : 72.1%
  frog    : 83.5%
  horse   : 81.8%
  ship    : 87.2%
  truck   : 93.0%
